# CerebroSpheroid V3: Zero-Bandwidth Google Colab Cloud Training

This notebook trains **CerebroSpheroid V3 (162M params)** on Google Colab (16 GB T4 or A100 GPU) with **minimal upload and download from your PC**:
- **Upload from PC**: Only the lightweight code bundle (~350 KB) or `git clone` (0 KB).
- **Datasets**: Downloaded directly inside Google Colab via cloud datacenter links (~100 MB/s, 0 KB PC bandwidth).
- **Checkpoints**: Mirrored directly to your Google Drive (`/content/drive/MyDrive/cerebro_checkpoints/`), so you never lose progress and never need to download huge 1.3 GB files to your PC unless desired.

In [ ]:
# 1. Confirm GPU accelerator and install dependencies
# Ensure Runtime -> Change runtime type -> T4 GPU is selected.
!nvidia-smi
!pip install -q torch transformers datasets pyarrow huggingface_hub

In [ ]:
# 2. Mount Google Drive for persistent checkpoint storage
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_CKPT_DIR = '/content/drive/MyDrive/cerebro_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoints will be mirrored to Google Drive: {DRIVE_CKPT_DIR}')

In [ ]:
# 3. Unpack Code (Upload reasoning_agent_code_only.zip (~350 KB) to /content)
import zipfile, glob

%cd /content
cands = glob.glob('/content/*code_only*.zip') + glob.glob('/content/*colab*.zip')
if not cands:
    # Fallback to cloning from git if zip not uploaded
    print('No zip found in /content. If your repo is public, you can clone directly:')
    # !git clone https://github.com/<your-username>/reasoning_agent.git /content/reasoning_agent
else:
    zip_path = cands[0]
    print(f'Unpacking {zip_path}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall('/content/reasoning_agent')

%cd /content/reasoning_agent
!mkdir -p checkpoints data/openhermes data/corpus data/lmsys_chat_1m/data

In [ ]:
# 4. Direct Cloud Download of Datasets (Google Cloud speed, 0 KB PC bandwidth)
# Downloads the 759 MB OpenHermes 2.5 parquet directly from Hugging Face in ~5 seconds:
import os
oh_path = 'data/openhermes/openhermes_2_5.parquet'
if not os.path.exists(oh_path) or os.path.getsize(oh_path) < 100_000_000:
    print('Downloading OpenHermes 2.5 (759 MB) directly to Colab...')
    !curl -L --progress-bar -o {oh_path} "https://huggingface.co/api/datasets/teknium/OpenHermes-2.5/parquet/default/train/0.parquet"
    print('Downloaded OpenHermes:', os.path.getsize(oh_path) / (1024**2), 'MB')

# Rebuild canonical world knowledge split
!python3 training/build_corpus.py

In [ ]:
# 5. Check if warm-start checkpoint exists in Google Drive
import shutil, glob
drive_ckpts = glob.glob(f'{DRIVE_CKPT_DIR}/*.pt')
init_arg = 'none'
if drive_ckpts:
    latest_drive = sorted(drive_ckpts)[-1]
    local_target = f'checkpoints/{os.path.basename(latest_drive)}'
    print(f'Found existing checkpoint in Drive: {latest_drive}. Copying to {local_target}...')
    shutil.copy2(latest_drive, local_target)
    init_arg = local_target
else:
    print('No prior checkpoint found in Drive. Starting fresh V3 training on T4 GPU.')

In [ ]:
# 6. Launch Scaled V3 Cloud Training
# On Colab T4 (16 GB VRAM), we can safely run batch_size=2 or 4 and full 128-token windows!
!python3 training/train_v3_full.py \
    --arch v3 \
    --max_steps 20000 \
    --probe_tokens 0 \
    --batch_size 2 \
    --grad_accum 4 \
    --window 128 \
    --eval_interval 200 \
    --save_every 200 \
    --save_prefix cerebro_spheroid_v3_colab \
    --init_from {init_arg}

# Mirror checkpoints back to Google Drive
!cp -f checkpoints/cerebro_spheroid_v3_colab* {DRIVE_CKPT_DIR}/
print('Checkpoints backed up to Google Drive!')

## 7. Interactive Cognitive Inspection
Test the trained V3 model directly inside Colab.

In [ ]:
from src.inference import ReasoningAgent
import glob

best_ckpts = glob.glob('checkpoints/*best*.pt')
target_ckpt = best_ckpts[0] if best_ckpts else None

if target_ckpt:
    print(f'Loading {target_ckpt}...')
    agent = ReasoningAgent(checkpoint_path=target_ckpt, arch='v3')
    prompts = [
        'What is the capital of France? Answer:',
        'What is the capital of Japan? Answer:',
        'What is the capital of India? Answer:',
        'What is the chemical formula for water? Answer:',
        'If 4 books cost $20, how much do 6 books cost? Answer:'
    ]
    for p in prompts:
        print(f'\nQ: {p}')
        print('A:', agent.generate(p, thinking_cycles=1, max_tokens=32, temperature=0.0))
    agent.stop_living_heartbeat()
else:
    print('No checkpoint found to evaluate.')